In [9]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

PATH_INDICES = 'csv_dashboard/BaseINDICES-2020-2025.csv'
PATH_ING_CHILE = 'csv_dashboard/todas_las_ingenierias_chile.csv'

COL_INST_INDICES = 'Nombre Institución'
COL_REG_INDICES = 'Nombre Region'
COL_CARRERA_INDICES = 'Carrera Genérica'

try:
    df_indices = pd.read_csv(PATH_INDICES, sep=';', encoding='utf-8')
except Exception:
    df_indices = pd.read_csv(PATH_INDICES, sep=',', encoding='utf-8')

cols_num_indices = ['Matrícula primer año hombres', 'Matrícula primer año mujeres', 'Vacantes', 'Matrícula Primer Año', 'Valor de arancel', 'Matrícula Total']
for col in cols_num_indices:
    if col in df_indices.columns:
        if df_indices[col].dtype == 'object':
            df_indices[col] = df_indices[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df_indices[col] = pd.to_numeric(df_indices[col], errors='coerce').astype('float32')

df_indices['Año'] = pd.to_numeric(df_indices['Año'], errors='coerce').fillna(2024).astype('int16')
df_ing_indices = df_indices[df_indices[COL_CARRERA_INDICES].str.contains('Ingeniería', case=False, na=False)].copy()

df_nacional = None
for sep_test in [';', ',']:
    for encoding_test in ['utf-8', 'latin-1', 'utf-16', 'cp1252']:
        try:
            df_test = pd.read_csv(PATH_ING_CHILE, sep=sep_test, encoding=encoding_test)
            if 'Institución' in df_test.columns:
                df_nacional = df_test
                print(f"Base nacional cargada con exito usando separador '{sep_test}' y encoding '{encoding_test}'")
                break
        except Exception:
            continue
    if df_nacional is not None:
        break

if df_nacional is None:
    raise ValueError("No se pudo estructurar correctamente el archivo todas_las_ingenierias_chile.csv. Verifica las columnas.")

cache_graficos = df_ing_indices.groupby(['Año', COL_CARRERA_INDICES])[['Matrícula primer año hombres', 'Matrícula primer año mujeres', 'Matrícula Total', 'Valor de arancel']].agg({
    'Matrícula primer año hombres': 'sum', 'Matrícula primer año mujeres': 'sum', 'Matrícula Total': 'sum', 'Valor de arancel': 'median'
}).reset_index()

cache_mapas = df_ing_indices.groupby(['Año', COL_REG_INDICES])[['Matrícula Total', 'Matrícula Primer Año', 'Valor de arancel']].agg({
    'Matrícula Total': 'sum', 'Matrícula Primer Año': 'sum', 'Valor de arancel': 'median'
}).reset_index()

cache_kpi_real = df_nacional.copy()

print("Bases sincronizadas con exito.")

Base nacional cargada con exito usando separador ';' y encoding 'utf-16'
Bases sincronizadas con exito.


In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import plotly.express as px

estilos_css = widgets.HTML("""
<style>
    .tabs-redondeadas .btn {
        border-radius: 16px !important; 
        margin-right: 6px !important;   
        border: 1px solid #bce8f1 !important;
    }
</style>
""")

tabs_navegacion = widgets.ToggleButtons(
    options=['KPIs', 'GRÁFICOS', 'MAPAS', 'ML (RANDOM FOREST PESOS)'],
    value='KPIs',
    button_style='info',
    layout=widgets.Layout(width='100%', margin='0px 0px 5px 0px')
)
tabs_navegacion.add_class('tabs-redondeadas') 

linea_separadora = widgets.HTML("<hr style='border: 0; border-top: 3px solid #000000; margin: 5px 0px 15px 0px; width: 100%; opacity: 1;'>")

selector_kpi_institucion = widgets.Dropdown(
    options=['---', 'Universidad de Chile', 'Pontificia Universidad Catolica', 'Universidad de Concepcion', 'Universidad Austral de Chile'],
    value='---',
    description='Institucion:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

selector_kpi_carrera = widgets.Dropdown(
    options=['---', 'Ingenieria Civil Informatica', 'Ingenieria Civil Industrial', 'Ingenieria Civil en Obras Civiles', 'Ingenieria Comercial'],
    value='---',
    description='Carrera:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='5px 0px 15px 0px')
)

panel_izquierdo_kpis = widgets.VBox([
    widgets.HTML("<h4>Filtros KPI</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_kpi_institucion,
    selector_kpi_carrera,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>"
                 "<i>Selecciona una institucion y una carrera especifica para cargar los indicadores unificados.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_kpi_cards = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_kpis = widgets.HBox([panel_izquierdo_kpis, area_kpi_cards], layout=widgets.Layout(width='100%', height='100%', padding='5px'))

diccionario_graficos = {
    '---': '',
    'Brecha de Genero en Matematicas (M1 vs M2)': 'csv/boxplot_genero_m1m2.png',
    'Distribucion por Rama Educacional (HC vs TP)': 'csv/violin_rama_educacional.png',
    'Evolucion del Puntaje por Dependencia': 'csv/lineplot_gap_evolution.png'
}

selector_graficos = widgets.Dropdown(
    options=list(diccionario_graficos.keys()),
    value='---',
    description='Grafico:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

panel_izquierdo_graficos = widgets.VBox([
    widgets.HTML("<h4>Reportes Estadisticos</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_graficos,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>"
                 "<i>Selecciona un reporte estadistico para cargar el analisis interactivo.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_imagen_grafico = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_graficos = widgets.HBox([panel_izquierdo_graficos, area_imagen_grafico], layout=widgets.Layout(width='100%', height='100%', padding='5px'))

diccionario_mapas = {
    '---': '',
    'Mapa de Empleabilidad Regional': 'csv/mapa_empleabilidad_2025.png',
    'Mapa de Retencion de Primer Año': 'csv/mapa_retencion_2025.png',
    'Mapa de Ingresos Promedio': 'csv/mapa_ingresos_2025.png'
}

selector_mapas = widgets.Dropdown(
    options=list(diccionario_mapas.keys()),
    value='---',
    description='Ver mapa:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

panel_izquierdo_mapas = widgets.VBox([
    widgets.HTML("<h4>Variables</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_mapas,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>"
                 "<i>Elige una opcion para actualizar la distribucion regional.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_imagen_mapa = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_mapas = widgets.HBox([panel_izquierdo_mapas, area_imagen_mapa], layout=widgets.Layout(width='100%', height='100%', padding='5px'))

layout_tab_ml = widgets.VBox([
    widgets.HTML("<div style='padding: 40px; text-align: center; color: #999; font-style: italic; font-family: sans-serif;'>"
                 "<h3>[ Pestaña de ML Limpia ]</h3>Espacio disponible para analisis predictivo.</div>")
], layout=widgets.Layout(width='100%', height='100%'))

contenedor_cuerpo = widgets.Output(layout=widgets.Layout(width='100%', height='430px', overflow='auto'))

print("Celda 2: Arquitectura visual cargada sin dimensiones.")

Celda 2: Arquitectura visual cargada sin dimensiones.


In [11]:
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

ANCHO_FIJO = '980px'
ALTO_FIJO = '580px'

def sueldo_a_numero(texto):
    if pd.isna(texto) or str(texto).lower() == 's/i': 
        return None
    t = str(texto).lower()
    if '3 millones 500' in t: return 3750000
    if '3 millones a' in t: return 3250000
    if '2 millones 500' in t: return 2750000
    if '2 millones 400' in t: return 2450000
    if '2 millones 300' in t: return 2350000
    if '2 millones 200' in t: return 2250000
    if '2 millones 100' in t: return 2150000
    if '2 millones' in t: return 2050000
    if '1 millón 900' in t: return 1950000
    if '1 millón 800' in t: return 1850000
    if '1 millón 700' in t: return 1750000
    if '1 millón 600' in t: return 1650000
    if '1 millón 500' in t: return 1550000
    if '1 millón 400' in t: return 1450000
    if '1 millón 300' in t: return 1350000
    if '1 millón 200' in t: return 1250000
    if '1 millón 100' in t: return 1150000
    if '1 millón' in t: return 1050000
    return None

def alternar_pestanas(change):
    with contenedor_cuerpo:
        clear_output(wait=True)
        pestana_activa = change['new'] if change else tabs_navegacion.value
        
        if pestana_activa == 'KPIs':
            display(layout_tab_kpis)
            actualizar_kpi_cards(None)
        elif pestana_activa == 'GRÁFICOS':
            display(layout_tab_graficos)
            actualizar_imagen_grafico(None)
        elif pestana_activa == 'MAPAS':
            display(layout_tab_mapas)
            actualizar_imagen_mapa(None)
        elif pestana_activa == 'ML (RANDOM FOREST PESOS)':
            display(layout_tab_ml)

tabs_navegacion.observe(alternar_pestanas, names='value')

def actualizar_kpi_cards(change):
    with area_kpi_cards:
        clear_output(wait=True)
        institucion = selector_kpi_institucion.value
        carrera = selector_kpi_carrera.value
        
        if institucion == '---' or carrera == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'>"
                                 "<h4>[ Selecciona una institucion y una carrera para desplegar las metricas KPI ]</h4></div>"))
        else:
            m_inst = institucion.lower().replace('pontificia ', '').split(' de ')[0]
            
            dic_carreras = {
                'Ingenieria Civil Informatica': 'informática|computación|software',
                'Ingenieria Civil Industrial': 'industrial',
                'Ingenieria Civil en Obras Civiles': 'obras civiles|civil',
                'Ingenieria Comercial': 'comercial'
            }
            m_carr = dic_carreras.get(carrera, 'invalid')
            
            df_res = cache_kpi_real[
                (cache_kpi_real['Institución'].str.lower().str.contains(m_inst, na=False)) &
                (cache_kpi_real['Carrera'].str.lower().str.contains(m_carr, na=False))
            ].copy()
            
            if not df_res.empty:
                for col in ['Retención de 1er año', 'Empleabilidad al 1er año', 'Empleabilidad al 2º Año']:
                    if col in df_res.columns:
                        df_res[col] = df_res[col].astype(str).str.replace('%', '', regex=False).str.replace(',', '.', regex=False)
                        df_res[col] = pd.to_numeric(df_res[col], errors='coerce')
                
                df_res['Duración Real (semestres)'] = pd.to_numeric(df_res['Duración Real (semestres)'], errors='coerce')
                df_res['Sueldo_Calculado'] = df_res['Ingreso promedio al 4° año'].apply(sueldo_a_numero)
                
                mean_ret = df_res['Retención de 1er año'].mean()
                v_ret = f"{mean_ret:.1f}%" if not np.isnan(mean_ret) else "--%"
                
                all_emp = pd.concat([df_res['Empleabilidad al 1er año'], df_res['Empleabilidad al 2º Año']])
                mean_emp = all_emp.mean()
                v_emp = f"{mean_emp:.1f}%" if not np.isnan(mean_emp) else "--%"
                
                mean_dur = df_res['Duración Real (semestres)'].mean()
                v_dur = f"{mean_dur:.1f} sem" if not np.isnan(mean_dur) else "-- sem"
                
                mean_ingreso = df_res['Sueldo_Calculado'].mean()
                v_ingreso = f"${int(mean_ingreso):,}".replace(',', '.') if not np.isnan(mean_ingreso) else "No disponible"
            else:
                v_ret, v_emp, v_dur, v_ingreso = "--%", "--%", "-- sem", "--"
                
            html_content = f"""
            <div style='font-family: sans-serif; padding: 5px; height:100%;'>
                <h4 style='color: #2c3e50; margin-top: 0; margin-bottom: 5px;'>Promedios Globales Unificados: {institucion}</h4>
                <h5 style='color: #555; margin-top: 0; margin-bottom: 15px; font-weight: normal;'>Programa Agrupado: <b>{carrera}</b></h5>
                
                <div style='display: flex; justify-content: space-between; gap: 10px; margin-bottom: 15px;'>
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5bc0de; padding: 10px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Retención Global</div>
                        <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_ret}</div>
                        <div style='font-size: 0.72em; color: #999; margin-top: 2px;'>Promedio historico</div>
                    </div>
                    
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5cb85c; padding: 10px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Empleabilidad Total</div>
                        <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_emp}</div>
                        <div style='font-size: 0.72em; color: #999; margin-top: 2px;'>Media unificada (1er y 2do año)</div>
                    </div>
                    
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #f0ad4e; padding: 10px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Duración Real Media</div>
                        <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_dur}</div>
                        <div style='font-size: 0.72em; color: #999; margin-top: 2px;'>Semestres efectivos</div>
                    </div>
                </div>
                
                <div style='background-color: #eef9f0; border: 1px solid #c3e6cb; border-left: 6px solid #28a745; padding: 12px; border-radius: 4px; margin-bottom: 15px; text-align: center;'>
                    <div style='font-size: 0.85em; color: #155724; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px;'>Sueldo Promedio Unificado de la Disciplina</div>
                    <div style='font-size: 1.8em; font-weight: bold; color: #1e7e34; margin-top: 5px;'>{v_ingreso}</div>
                </div>
            </div>
            """
            display(widgets.HTML(html_content))

selector_kpi_institucion.observe(actualizar_kpi_cards, names='value')
selector_kpi_carrera.observe(actualizar_kpi_cards, names='value')

def actualizar_imagen_mapa(change):
    with area_imagen_mapa:
        clear_output(wait=True)
        opcion_seleccionada = selector_mapas.value
        
        if opcion_seleccionada == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'>"
                                 "<h4>[ Elige una opción en el menú izquierdo para renderizar el reporte regional ]</h4></div>"))
        else:
            df_geo = cache_mapas[cache_mapas['Año'] == cache_mapas['Año'].max()]
            
            if opcion_seleccionada == 'Mapa de Empleabilidad Regional':
                fig = px.bar(df_geo.sort_values('Matrícula Total', ascending=True), x='Matrícula Total', y=COL_REG_INDICES, orientation='h', color='Matrícula Total', color_continuous_scale='Viridis', labels={'Matrícula Total': 'Estudiantes Activos', COL_REG_INDICES: 'Región'})
            elif opcion_seleccionada == 'Mapa de Retencion de Primer Año':
                fig = px.bar(df_geo.sort_values('Matrícula Primer Año', ascending=True), x='Matrícula Primer Año', y=COL_REG_INDICES, orientation='h', color='Matrícula Primer Año', color_continuous_scale='Plasma', labels={'Matrícula Primer Año': 'Mechones Ingresados', COL_REG_INDICES: 'Región'})
            elif opcion_seleccionada == 'Mapa de Ingresos Promedio':
                fig = px.bar(df_geo.sort_values('Valor de arancel', ascending=True), x='Valor de arancel', y=COL_REG_INDICES, orientation='h', color='Valor de arancel', color_continuous_scale='coolwarm', labels={'Valor de arancel': 'Mediana Arancel', COL_REG_INDICES: 'Región'})
                fig.update_layout(xaxis_tickformat="$")
                
            fig.update_layout(height=380, width=590, margin=dict(l=10, r=10, t=30, b=10), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
            fig.show()

selector_mapas.observe(actualizar_imagen_mapa, names='value')

def actualizar_imagen_grafico(change):
    with area_imagen_grafico:
        clear_output(wait=True)
        opcion_seleccionada = selector_graficos.value
        
        if opcion_seleccionada == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'>"
                                 "<h4>[ Selecciona un reporte del panel izquierdo para desplegar el grafico ]</h4></div>"))
        else:
            if opcion_seleccionada == 'Brecha de Genero en Matematicas (M1 vs M2)':
                df_g = cache_graficos.groupby('Año')[['Matrícula primer año hombres', 'Matrícula primer año mujeres']].sum().reset_index()
                df_g['Porcentaje Mujeres (%)'] = (df_g['Matrícula primer año mujeres'] / (df_g['Matrícula primer año hombres'] + df_g['Matrícula primer año mujeres'])) * 100
                fig = px.line(df_g, x='Año', y='Porcentaje Mujeres (%)', markers=True)
                fig.update_yaxes(range=[0, 50])
            elif opcion_seleccionada == 'Distribucion por Rama Educacional (HC vs TP)':
                df_g = cache_graficos.groupby('Año')['Matrícula Total'].sum().reset_index()
                fig = px.line(df_g, x='Año', y='Matrícula Total', markers=True, labels={'Matrícula Total': 'Comunidad Total Estudiantes'})
            elif opcion_seleccionada == 'Evolucion del Puntaje por Dependencia':
                df_g = cache_graficos.groupby('Año')['Valor de arancel'].median().reset_index()
                fig = px.line(df_g, x='Año', y='Valor de arancel', markers=True, labels={'Valor de arancel': 'Arancel Mediano ($)'})
                fig.update_layout(yaxis_tickformat="$")
            else:
                display(widgets.HTML("<div style='padding:20px;'><h4>Reporte No Soportado</h4></div>"))
                return

            fig.update_layout(height=380, width=590, margin=dict(l=10, r=10, t=30, b=10), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
            fig.show()

selector_graficos.observe(actualizar_imagen_grafico, names='value')

txt_usuario = widgets.Text(description='Usuario:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
txt_password = widgets.Password(description='Clave:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
btn_login = widgets.Button(description='Autenticar', button_style='primary', icon='lock', layout=widgets.Layout(margin='20px 0px 5px 0px', width='280px'))
html_feedback = widgets.HTML(value="")

formulario_interno = widgets.VBox([
    widgets.HTML("<h3 style='text-align: center; font-family: sans-serif; color: #333; margin-top:0;'>SISTEMA DE ACCESO</h3><hr style='width: 100%; border: 0; border-top: 1px solid #ccc;'>"),
    txt_usuario, txt_password, btn_login, html_feedback
], layout=widgets.Layout(width='360px', padding='25px', border='1px solid #ccc', bg_color='#ffffff', align_items='center', border_radius='4px'))

cuadro_login = widgets.VBox([formulario_interno], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', bg_color='#f4f4f4', justify_content='center', align_items='center'))
dashboard_final = widgets.VBox([estilos_css, tabs_navegacion, linea_separadora, contenedor_cuerpo], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', bg_color='#f4f4f4', padding='20px'))

def validar_credenciales(b):
    if txt_usuario.value == 'admin' and txt_password.value == 'admin':
        with lienzo_maestro:
            clear_output()
            display(dashboard_final)
            alternar_pestanas(None)
    else:
        txt_password.value = ""
        html_feedback.value = "<div style='color: #d9534f; font-weight: bold; text-align: center; margin-top: 12px; font-family: sans-serif;'>Error: Credenciales Incorrectas</div>"

btn_login.on_click(validar_credenciales)
lienzo_maestro = widgets.Output()

print("Celda 3: Motores reactivos unificados ejecutados con exito.")

Celda 3: Motores reactivos unificados ejecutados con exito.


In [12]:
# %%
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 4 — Inicialización y Lanzamiento en Pantalla       ║
# ╚══════════════════════════════════════════════════════════╝

# 1. Desplegamos el nodo de salida raíz (el lienzo maestro) en Jupyter
display(lienzo_maestro)

# 2. Forzamos el renderizado inicial de la caja de login dentro del lienzo
with lienzo_maestro:
    clear_output()
    display(cuadro_login)

Output()